# Run TemGX

Run this notebook for the selected `DATASET` / `MODEL` combo.

This notebook:
- Runs the official `TemGX` implementation from the vendored TemGX submodule.
- Loads the matching checkpoint and explain index for the chosen combo.
- Saves official run artifacts under `resources/results/official_temgx/` and finishes with the shared one-spot metrics summary.

Usage:
- Change `DATASET` and `MODEL` in the first code cell when needed.
- Run the notebook top to bottom.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

_PROJECT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _PROJECT_CANDIDATES if (p / "I_explainer_benchmark").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from I_explainer_benchmark.src.notebooks.explainer_notebook_setup import (
    initialize_explainer_notebook,
    prepare_explainer_run,
)

# Select the dataset / model combo here.
DATASET = "simulate_v1"
MODEL = "tgn"

# Shared notebook setup. Leave the rest unchanged.
BOOT = initialize_explainer_notebook("04_temgx.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)
NB = BOOT.env
CFG = BOOT.config
SETTINGS = BOOT.settings
BENCHMARK_ROOT = BOOT.bench_root
REPO_ROOT = BOOT.repo_root


In [2]:
import torch
from I_explainer_benchmark.src.notebooks.notebook_helpers import set_global_seed

MODEL = str(MODEL).strip().lower()
SUPPORTED_MODEL_TYPES = set(CFG["supported_model_types"])
if MODEL not in SUPPORTED_MODEL_TYPES:
    raise ValueError(f"Unsupported MODEL={MODEL!r}. Choose one of {sorted(SUPPORTED_MODEL_TYPES)}")

CTX = prepare_explainer_run("04_temgx.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)

DATASET_NAME = CTX.dataset
MODEL_TYPE = CTX.model
SEED = int(CFG["seed"])
set_global_seed(SEED)

N_EVAL_EVENTS = int(SETTINGS["n_eval_events"])
CANDIDATES_SIZE = int(SETTINGS["candidates_size"])
USE_CACHED_TEMGX_RESULTS = bool(SETTINGS["use_cached_temgx_results"])
EXPLAIN_INDEX_PATH = CTX.explain_index_path
CKPT_PATH = CTX.checkpoint_path
DEVICE = CTX.device
out_dir = BENCHMARK_ROOT / "resources" / "results" / str(CFG["out_dir_name"])
out_dir.mkdir(parents=True, exist_ok=True)

print("DATASET:", DATASET_NAME)
print("MODEL  :", MODEL_TYPE)
print("CKPT   :", CKPT_PATH)
print("DEVICE :", DEVICE)
print("OUT    :", out_dir)


DATASET: simulate_v1
MODEL  : tgn
CKPT   : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/simulate_v1/tgn/tgn_simulate_v1_best.pth
DEVICE : mps
OUT    : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_temgx


In [3]:
# Run/reuse TemGX explanations and persist results.jsonl for downstream metrics
from pathlib import Path

import pandas as pd

from I_explainer_benchmark.src.explainers.adapters.temgx_adapter import _load_temgx_module
from I_explainer_benchmark.src.explainers.builder import make_explainer_builder
from I_explainer_benchmark.src.notebooks.explainer_notebook_runtime import (
    load_jsonl_records,
    prepare_standard_edge_explainer_run,
    run_or_reuse_explanations,
)
from I_explainer_benchmark.src.notebooks.notebook_helpers import forced_target_event_ids_for_combo as _forced_target_event_ids_for_combo

# Hard-guard: ensure we are using the TemGX submodule implementation.
temgx_mod = _load_temgx_module()
temgx_src = Path(temgx_mod.__file__).resolve()
temgx_src_norm = str(temgx_src).replace('\\', '/')
print('TemGX source:', temgx_src)
assert '/submodules/explainer/TemGX/link/scripts/temgx.py' in temgx_src_norm, (
    'Expected TemGX implementation from submodules/explainer/TemGX/link/scripts/temgx.py'
)

FORCED_TARGET_EVENT_IDS = list(
    globals().get('FORCED_TARGET_EVENT_IDS', _forced_target_event_ids_for_combo(DATASET_NAME, MODEL_TYPE))
)

build_explainer = make_explainer_builder(
    dataset_name=DATASET_NAME,
    model_type=MODEL_TYPE,
    device=DEVICE,
    seed=SEED,
    callable_scope=globals(),
)

temgx_explainer = build_explainer(
    'temgx',
    allow_missing=False,
    overrides={
        'alias': 'temgx',
        'seed': SEED,
        'cache': True,
    },
)
print('Explainer alias:', temgx_explainer.alias)

ctx = prepare_standard_edge_explainer_run(
    dataset_name=DATASET_NAME,
    model_name=MODEL_TYPE,
    ckpt_path=CKPT_PATH,
    device=DEVICE,
    explain_index_path=EXPLAIN_INDEX_PATH,
    n_eval_events=N_EVAL_EVENTS,
    out_dir=out_dir,
    explainer=temgx_explainer,
    candidates_size=CANDIDATES_SIZE,
    seed=SEED,
    include_event_ids=FORCED_TARGET_EVENT_IDS,
)
model = ctx.model
events = ctx.events
backbone = model.backbone
all_event_idxs = ctx.all_event_idxs
target_event_idxs = ctx.target_event_idxs
anchors = ctx.anchors

print('Anchors selected:', len(anchors), '/', len(all_event_idxs))
if anchors:
    print('First target event_idx:', anchors[0]['event_idx'])

out, run_dir, out_jsonl, base_name = run_or_reuse_explanations(
    runner=ctx.runner,
    anchors=anchors,
    dataset_name=DATASET_NAME,
    model_name=MODEL_TYPE,
    explainer_name='temgx',
    target_event_idxs=target_event_idxs,
    use_cached=USE_CACHED_TEMGX_RESULTS,
)

print('run_dir:', run_dir)
print('out_jsonl:', out_jsonl)
print('base_name:', base_name)

_records = load_jsonl_records(out_jsonl)

temgx_rows = []
for rec in _records:
    ctx_row = rec.get('context') or {}
    tgt = ctx_row.get('target') or {}
    res = rec.get('result') or {}
    extras = res.get('extras') or {}
    cand = list(extras.get('candidate_eidx') or [])
    selected = list(extras.get('selected_eidx') or [])
    stats = extras.get('temgx_stats') or {}
    temgx_rows.append({
        'event_idx': int(tgt.get('event_idx')) if tgt.get('event_idx') is not None else pd.NA,
        'candidate_size': int(len(cand)),
        'selected_size': int(len(selected)),
        'counterfactual_prediction': pd.to_numeric(extras.get('counterfactual_prediction'), errors='coerce'),
        'has_temgx_stats': bool(isinstance(stats, dict) and len(stats) > 0),
        'elapsed_sec': pd.to_numeric(res.get('elapsed_sec'), errors='coerce'),
    })

temgx_results_df = pd.DataFrame(temgx_rows)
out_csv = out_dir / f'{base_name}.csv'
temgx_results_df.to_csv(out_csv, index=False)

print('Saved explanations:', out_csv)
print(temgx_results_df.head())


TemGX source: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/submodules/explainer/TemGX/link/scripts/temgx.py
Built explainer 'temgx' from /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/configs/explainer/temgx.json
Explainer alias: temgx
102 events to explain
Anchors selected: 100 / 102
First target event_idx: 3
Using cached temgx explanations from: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_temgx/simulate_v1_tgn_official_temgx_20260315_012452
run_dir: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_temgx/simulate_v1_tgn_official_temgx_20260315_012452
out_jsonl: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_

In [4]:
# Shared metrics + export pipeline (clean notebook wrapper)
from I_explainer_benchmark.src.notebooks.notebook_metrics_pipeline import run_notebook_metrics_from_namespace

_pipeline_out = run_notebook_metrics_from_namespace(globals(), CFG)
globals().update(_pipeline_out)
run_dir_export = _pipeline_out['run_dir_export']
export_root = _pipeline_out['export_root']
out_jsonl = _pipeline_out['out_jsonl']
out_dir = _pipeline_out['out_dir']
base_name = _pipeline_out['base_name']
print('CURRENT_RUN_ID:', _pipeline_out['CURRENT_RUN_ID'])
print('Saved run export directory:', run_dir_export)
print('Updated global tables under:', export_root)


CURRENT_RUN_ID: simulate_v1_tgn_official_temgx_20260315_012452
Saved run export directory: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready/simulate_v1_tgn_official_temgx_20260315_012452
Updated global tables under: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready


In [5]:
# One-spot metrics summary (official)
from I_explainer_benchmark.src.notebooks.notebook_display import show_one_spot_metrics_summary

_one_spot = globals().get("one_spot", globals().get("_pipeline_out", {}).get("one_spot", {}))
show_one_spot_metrics_summary(_one_spot)


One-spot metrics summary (official):


,dataset,model,explainer,variant,n_events,best_fid,best_fid_sparsity,aufsc,best_minus_aufsc,best_fid_raw,...,tempme_acc_auc.ratio_acc,tempme_acc_auc.ratio_prob,tempme_acc_auc.ratio_logit,tempme_acc_auc.ratio_aps,tempme_acc_auc.ratio_auc,temgx_aufsc,cody_AUFSC_plus,cody_AUFSC_minus,cody_CHARR,elapsed_sec
0,simulate_v1,tgn,temgx,official,100,1.1637848212,1.0,-0.3186056283,1.4823904495,1.1637848212,...,0.7365625,-0.0143587992,-0.2024367351,0.809375,0.9583333333,0.1285051505,0.3756839257,0.5993160743,0.4618531601,0.0284524234
